In [ ]:
import pandas as pd
import plotly.graph_objects as go
import numpy as np

In [ ]:
day = 1
df = pd.read_csv(f"./round-2-island-data-bottle/prices_round_2_day_{day}.csv", sep=";", header=0)

In [ ]:
df

In [ ]:
df = df[['timestamp', 'ORCHIDS', 'TRANSPORT_FEES', 'EXPORT_TARIFF', 'IMPORT_TARIFF','SUNLIGHT', 'HUMIDITY']]

In [ ]:
df

In [ ]:
df.loc[:, 'ORCHIDS'] = df['ORCHIDS'].ewm(alpha = 0.05).mean().reset_index(drop=True)

In [ ]:
df['HUMIDITY_DIFF'] = np.abs(df["HUMIDITY"] - 70)

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Create a subplot figure with separate y-axes for each feature
fig = make_subplots(specs=[[{"secondary_y": True}]])

# Add traces for each feature
fig.add_trace(go.Scatter(x=df['timestamp'], y=df['ORCHIDS'], name='ORCHIDS'), secondary_y=False)
fig.add_trace(go.Scatter(x=df['timestamp'], y=df['TRANSPORT_FEES'], name='TRANSPORT_FEES'), secondary_y=True)
fig.add_trace(go.Scatter(x=df['timestamp'], y=df['EXPORT_TARIFF'], name='EXPORT_TARIFF'), secondary_y=True)
fig.add_trace(go.Scatter(x=df['timestamp'], y=df['IMPORT_TARIFF'], name='IMPORT_TARIFF'), secondary_y=True)
fig.add_trace(go.Scatter(x=df['timestamp'], y=df['SUNLIGHT'], name='SUNLIGHT'), secondary_y=True)
fig.add_trace(go.Scatter(x=df['timestamp'], y=df['HUMIDITY'], name='HUMIDITY'), secondary_y=True)
fig.add_trace(go.Scatter(x=df['timestamp'], y=df['HUMIDITY_DIFF'], name='HUMIDITY_DIFF'), secondary_y=True)

# Set the layout and axis properties
fig.update_layout(
    title='Feature Values over Time',
    xaxis_title='Timestamp',
    yaxis_title='ORCHIDS',
    legend=dict(x=0, y=1.15, orientation='h')
)

# Update y-axis titles
fig.update_yaxes(title_text="ORCHIDS", secondary_y=False)
fig.update_yaxes(title_text="Other Features", secondary_y=True)

# Show the plot
fig.show()

#127.6, 171.1

negative corr with transport fees 
mayyyyybe positive corr with export tariff?? looks shit tho 
honestly tarrifs possibly uncorr.
possibly uncorr with sunlight...if our calc is right sunlight is just nominal per day 


transportation fees perhaps lag orchids going up–could be since ducks probably set price and they know when transportation fees are going up??? but price sticky on the way down?? 




In [ ]:
def get_prev_returns(df, col, its):
    prev_col = f"{col}_prev_{its}_its"
    df[prev_col] = df[col].shift(its)
    
    if col == 'HUMIDITY_DIFF':
        df[f"{col}_returns_from_{its}_its_ago"] = ((df[col] - df[prev_col]) / df[prev_col]).where(df['HUMIDITY_DIFF'] >= 10, 0)
    else:
        df[f"{col}_returns_from_{its}_its_ago"] = (df[col] - df[prev_col]) / df[prev_col]
    
    df.drop(columns=[prev_col], inplace=True)
    return df

def get_future_returns(df, col, its):
    future_col = f"{col}_future_{its}_its"
    df[future_col] = df[col].shift(-its)
    
    if col == 'HUMIDITY_DIFF':
        df[f"{col}_returns_in_{its}_its"] = ((df[future_col] - df[col]) / df[col]).where(df['HUMIDITY_DIFF'] >= 7.5, 0)
    else:
        df[f"{col}_returns_in_{its}_its"] = (df[future_col] - df[col]) / df[col]
    
    df.drop(columns=[future_col], inplace=True)
    return df

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

def generate_returns_dataframe(df, iteration):
    columns_to_process = ['ORCHIDS', 'TRANSPORT_FEES', 'EXPORT_TARIFF', 'IMPORT_TARIFF', 'SUNLIGHT', 'HUMIDITY', 'HUMIDITY_DIFF']
    new_df = df.copy()

    for col in columns_to_process:
        new_df = get_prev_returns(new_df, col, iteration)
        if col == 'ORCHIDS':
            new_df = get_future_returns(new_df, col, iteration)

    return new_df

def calculate_correlation(df, iteration):
    columns_to_process = ['TRANSPORT_FEES', 'EXPORT_TARIFF', 'IMPORT_TARIFF', 'SUNLIGHT', 'HUMIDITY', 'HUMIDITY_DIFF']
    correlations = {}

    for col in columns_to_process:
        corr = df[f"{col}_returns_from_{iteration}_its_ago"].corr(df[f"ORCHIDS_returns_in_{iteration}_its"], method='pearson')
        correlations[col] = corr

    return correlations

# Assuming you have your original DataFrame named 'df'
iteration_candidates = [1, 5, 10, 50, 100, 250, 500]

for iteration in iteration_candidates:
    print(f"Iteration: {iteration}")
    new_df = generate_returns_dataframe(df, iteration)
    correlations = calculate_correlation(new_df, iteration)

    # Print the correlation values
    for col, corr in correlations.items():
        print(f"Correlation between {col}_returns_from_{iteration}_its_ago and ORCHIDS_returns_in_{iteration}_its: {corr}")

    # Create a bar graph of the correlations
    plt.figure(figsize=(10, 6))
    plt.bar(correlations.keys(), correlations.values())
    plt.xlabel('Features')
    plt.ylabel('Correlation')
    plt.title(f"Correlation with ORCHIDS_returns_in_{iteration}_its")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

def generate_returns_dataframe(df, iteration):
    columns_to_process = ['ORCHIDS', 'TRANSPORT_FEES', 'EXPORT_TARIFF', 'IMPORT_TARIFF', 'SUNLIGHT', 'HUMIDITY', 'HUMIDITY_DIFF']
    new_df = df.copy()

    for col in columns_to_process:
        new_df = get_prev_returns(new_df, col, iteration)
        if col == 'ORCHIDS':
            new_df = get_future_returns(new_df, col, iteration)

    return new_df

def calculate_correlation(df, iteration):
    columns_to_process = ['TRANSPORT_FEES', 'EXPORT_TARIFF', 'IMPORT_TARIFF', 'SUNLIGHT', 'HUMIDITY', 'HUMIDITY_DIFF']
    correlations = {}

    for col in columns_to_process:
        corr = df[f"{col}_returns_from_{iteration}_its_ago"].corr(df[f"ORCHIDS_returns_in_{iteration}_its"], method='pearson')
        correlations[col] = corr

    return correlations

# Assuming you have your original DataFrame named 'df'
iteration_candidates = [1, 5, 10, 50, 100, 250, 500]

for iteration in iteration_candidates:
    print(f"Iteration: {iteration}")
    new_df = generate_returns_dataframe(df, iteration)
    correlations = calculate_correlation(new_df, iteration)

    # Print the correlation values
    for col, corr in correlations.items():
        print(f"Correlation between {col}_returns_from_{iteration}_its_ago and ORCHIDS_returns_in_{iteration}_its: {corr}")

    # Create a bar graph of the correlations
    plt.figure(figsize=(10, 6))
    plt.bar(correlations.keys(), correlations.values())
    plt.xlabel('Features')
    plt.ylabel('Correlation')
    plt.title(f"Correlation with ORCHIDS_returns_in_{iteration}_its")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

def generate_returns_dataframe(df, iteration):
    columns_to_process = ['ORCHIDS', 'TRANSPORT_FEES', 'EXPORT_TARIFF', 'IMPORT_TARIFF', 'SUNLIGHT', 'HUMIDITY', 'HUMIDITY_DIFF']
    new_df = df.copy()

    for col in columns_to_process:
        new_df = get_prev_returns(new_df, col, iteration)
        new_df = get_future_returns(new_df, col, iteration)

    return new_df

def calculate_correlation(df, iteration):
    columns_to_process = ['ORCHIDS', 'TRANSPORT_FEES', 'EXPORT_TARIFF', 'IMPORT_TARIFF', 'SUNLIGHT', 'HUMIDITY', 'HUMIDITY_DIFF']
    correlations = {}

    for col in columns_to_process:
        corr = df[f"{col}_returns_from_{iteration}_its_ago"].corr(df[f"ORCHIDS_returns_in_{iteration}_its"], method='pearson')
        correlations[col] = corr

    return correlations

# Assuming you have your original DataFrame named 'df'
iteration_candidates = [100, 150, 200, 250, 300, 400, 500, 700, 1000, 1200, 1500, 2000]
correlation_data = {}

for iteration in iteration_candidates:
    print(f"Iteration: {iteration}")
    new_df = generate_returns_dataframe(df, iteration)
    correlations = calculate_correlation(new_df, iteration)
    correlation_data[iteration] = correlations

    # Print the correlation values
    for col, corr in correlations.items():
        print(f"Correlation between {col}_returns_from_{iteration}_its_ago and ORCHIDS_returns_in_{iteration}_its: {corr}")

# Create a grid of bar charts
num_iterations = len(iteration_candidates)
num_cols = 2
num_rows = (num_iterations + num_cols - 1) // num_cols

fig, axes = plt.subplots(num_rows, num_cols, figsize=(12, 6 * num_rows), sharex=True, sharey=True)
axes = axes.flatten()

for i, iteration in enumerate(iteration_candidates):
    ax = axes[i]
    correlations = correlation_data[iteration]
    ax.bar(correlations.keys(), correlations.values())
    ax.set_xlabel('Features')
    ax.set_ylabel('Correlation')
    ax.set_title(f"Correlation with ORCHIDS_returns_in_{iteration}_its")
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

# Create a heatmap
heatmap_data = pd.DataFrame(correlation_data).T
plt.figure(figsize=(10, 8))
sns.heatmap(heatmap_data, annot=True, cmap='coolwarm', fmt='.2f')
plt.xlabel('Features')
plt.ylabel('Timeframes')
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

def generate_returns_dataframe(df, iteration):
    columns_to_process = ['ORCHIDS', 'TRANSPORT_FEES', 'EXPORT_TARIFF', 'IMPORT_TARIFF', 'SUNLIGHT', 'HUMIDITY', 'HUMIDITY_DIFF']
    new_df = df.copy()

    for col in columns_to_process:
        new_df = get_prev_returns(new_df, col, iteration)
        if col == 'ORCHIDS':
            new_df = get_future_returns(new_df, col, 500)  # Standardize to 500 iterations for ORCHIDS' future returns

    return new_df

def calculate_correlation(df, iteration):
    columns_to_process = ['ORCHIDS', 'TRANSPORT_FEES', 'EXPORT_TARIFF', 'IMPORT_TARIFF', 'SUNLIGHT', 'HUMIDITY', 'HUMIDITY_DIFF']
    correlations = {}

    for col in columns_to_process:
        corr = df[f"{col}_returns_from_{iteration}_its_ago"].corr(df["ORCHIDS_returns_in_500_its"], method='pearson')
        correlations[col] = corr

    return correlations

# Assuming you have your original DataFrame named 'df'
iteration_candidates = [100, 150, 200, 250, 300, 400, 500, 700, 1000, 1200, 1500, 2000]
correlation_data = {}

for iteration in iteration_candidates:
    print(f"Iteration: {iteration}")
    new_df = generate_returns_dataframe(df, iteration)
    correlations = calculate_correlation(new_df, iteration)
    correlation_data[iteration] = correlations

    # Print the correlation values
    for col, corr in correlations.items():
        print(f"Correlation between {col}_returns_from_{iteration}_its_ago and ORCHIDS_returns_in_500_its: {corr}")

# Create a grid of bar charts
num_iterations = len(iteration_candidates)
num_cols = 2
num_rows = (num_iterations + num_cols - 1) // num_cols

fig, axes = plt.subplots(num_rows, num_cols, figsize=(12, 6 * num_rows), sharex=True, sharey=True)
axes = axes.flatten()

for i, iteration in enumerate(iteration_candidates):
    ax = axes[i]
    correlations = correlation_data[iteration]
    ax.bar(correlations.keys(), correlations.values())
    ax.set_xlabel('Features')
    ax.set_ylabel('Correlation')
    ax.set_title(f"Correlation with ORCHIDS_returns_in_500_its (Past {iteration} iterations)")
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

# Create a heatmap
heatmap_data = pd.DataFrame(correlation_data).T
plt.figure(figsize=(10, 8))
sns.heatmap(heatmap_data, annot=True, cmap='coolwarm', fmt='.2f')
plt.xlabel('Features')
plt.ylabel('Past Iterations')
plt.title("Correlation Heatmap (Target: ORCHIDS_returns_in_500_its)")
plt.tight_layout()
plt.show()

linear regression: 

target: orchids returns in 500 its

features:
- orchids_returns in last 700 its
- export tariff last 500 its 
- humidity_diff last 700 its 
- 

In [ ]:
df_humidity_over_10 = df[df['HUMIDITY_DIFF'] >= 10]

In [ ]:
df_humidity_over_10

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

def generate_returns_dataframe(df, iteration):
    columns_to_process = ['ORCHIDS', 'HUMIDITY_DIFF']
    new_df = df.copy()

    for col in columns_to_process:
        new_df = get_prev_returns(new_df, col, iteration)
        new_df = get_future_returns(new_df, col, iteration)  # Use the same iteration for both HUMIDITY_DIFF and ORCHIDS

    return new_df

def calculate_correlation(df, iteration):
    columns_to_process = ['HUMIDITY_DIFF']
    correlations = {}

    for col in columns_to_process:
        corr = df[f"{col}_returns_from_{iteration}_its_ago"].corr(df[f"ORCHIDS_returns_in_{iteration}_its"], method='pearson')
        correlations[col] = corr

    return correlations

# Assuming you have your original DataFrame named 'df'
iteration_candidates = [10, 50, 75, 87, 100, 150, 200, 250, 300, 400, 500, 700, 1000, 1200, 1500, 2000]
correlation_data = {}

for iteration in iteration_candidates:
    print(f"Iteration: {iteration}")
    new_df = generate_returns_dataframe(df, iteration)
    new_df = new_df[new_df['HUMIDITY_DIFF'] > 10]  # Restrict to only when HUMIDITY_DIFF > 10
    correlations = calculate_correlation(new_df, iteration)
    correlation_data[iteration] = correlations

    # Print the correlation values
    for col, corr in correlations.items():
        print(f"Correlation between {col}_returns_from_{iteration}_its_ago and ORCHIDS_returns_in_{iteration}_its: {corr}")

# Create a line plot of the correlations
plt.figure(figsize=(10, 6))
plt.plot(iteration_candidates, [correlation_data[i]['HUMIDITY_DIFF'] for i in iteration_candidates], marker='o')
plt.xlabel('Iterations')
plt.ylabel('Correlation')
plt.title("Correlation between HUMIDITY_DIFF and ORCHIDS (HUMIDITY_DIFF > 10)")
plt.xticks(rotation=45)
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
from sklearn.linear_model import LinearRegression

def generate_returns_dataframe(df, iteration):
    columns_to_process = ['ORCHIDS', 'HUMIDITY_DIFF']
    new_df = df.copy()

    for col in columns_to_process:
        new_df = get_prev_returns(new_df, col, iteration)
        new_df = get_future_returns(new_df, col, iteration)

    return new_df

# Assuming you have your original DataFrame named 'df'
iteration = 100

# Generate the returns DataFrame
new_df = generate_returns_dataframe(df, iteration).dropna()

# Restrict the DataFrame to HUMIDITY_DIFF >= 10
new_df = new_df[new_df['HUMIDITY_DIFF'] >= 10]

# Prepare the input features (X) and target variable (y) for linear regression
X = new_df[f"HUMIDITY_DIFF_returns_from_{iteration}_its_ago"].values.reshape(-1, 1)
y = new_df[f"ORCHIDS_returns_in_{iteration}_its"].values.reshape(-1, 1)

# Create a linear regression model with no constant term
model = LinearRegression(fit_intercept=False)

# Fit the model
model.fit(X, y)

# Print the coefficient (slope) of the linear regression
coefficient = model.coef_[0][0]
print(f"Coefficient (Slope) of Linear Regression: {coefficient}")

In [ ]:
new_df

In [ ]:

df_day_0 = pd.read_csv(f"./round-2-island-data-bottle/prices_round_2_day_{0}.csv", sep=";", header=0)
df_day_1 = pd.read_csv(f"./round-2-island-data-bottle/prices_round_2_day_{1}.csv", sep=";", header=0)

In [ ]:
df_day_0 = df_day_0[['timestamp', 'ORCHIDS', 'TRANSPORT_FEES', 'EXPORT_TARIFF', 'IMPORT_TARIFF','SUNLIGHT', 'HUMIDITY']]
df_day_0.loc[:, 'ORCHIDS'] = df_day_0['ORCHIDS'].ewm(alpha = 0.05).mean().reset_index(drop=True)
df_day_0['HUMIDITY_DIFF'] = np.abs(df_day_0["HUMIDITY"] - 70)

In [ ]:
df_day_1 = df_day_1[['timestamp', 'ORCHIDS', 'TRANSPORT_FEES', 'EXPORT_TARIFF', 'IMPORT_TARIFF','SUNLIGHT', 'HUMIDITY']]
df_day_1.loc[:, 'ORCHIDS'] = df_day_1['ORCHIDS'].ewm(alpha = 0.05).mean().reset_index(drop=True)
df_day_1['HUMIDITY_DIFF'] = np.abs(df_day_1["HUMIDITY"] - 70)

In [ ]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

def generate_returns_dataframe(df, iteration):
    columns_to_process = ['ORCHIDS', 'HUMIDITY_DIFF']
    new_df = df.copy()

    for col in columns_to_process:
        new_df = get_prev_returns(new_df, col, iteration)
        new_df = get_future_returns(new_df, col, iteration)

    return new_df

# Assuming you have your dataframes named 'df_day_0' and 'df_day_1'
iteration = 100

# Generate the returns DataFrame for df_day_0
df_day_0_returns = generate_returns_dataframe(df_day_0, iteration).dropna()
df_day_0_returns = df_day_0_returns[df_day_0_returns['HUMIDITY_DIFF'] >= 15]

# Generate the returns DataFrame for df_day_1
df_day_1_returns = generate_returns_dataframe(df_day_1, iteration).dropna()
df_day_1_returns = df_day_1_returns[df_day_1_returns['HUMIDITY_DIFF'] >= 15]

# Concatenate the two DataFrames vertically
combined_df = pd.concat([df_day_0_returns, df_day_1_returns], ignore_index=True)

# Prepare the input features (X) and target variable (y) for linear regression
X = combined_df[f"HUMIDITY_DIFF_returns_from_{iteration}_its_ago"].values.reshape(-1, 1)
y = combined_df[f"ORCHIDS_returns_in_{iteration}_its"].values.reshape(-1, 1)

# Create a linear regression model with no constant term
model = LinearRegression(fit_intercept=False)

# Fit the model
model.fit(X, y)

# Get the coefficient (slope) of the linear regression
coefficient = model.coef_[0][0]

# Generate the equation
equation = f"ORCHIDS_returns_in_{iteration}_its = {coefficient:.4f} * HUMIDITY_DIFF_returns_from_{iteration}_its_ago"
print("Equation:")
print(equation)

# Make predictions using the fitted model
y_pred = model.predict(X)

# Calculate mean squared error (MSE)
mse = mean_squared_error(y, y_pred)
print(f"\nMean Squared Error (MSE): {mse:.4f}")

# Calculate R-squared value
r2 = r2_score(y, y_pred)
print(f"R-squared (R^2) Value: {r2:.4f}")

In [ ]:
df_day_neg_1 = pd.read_csv(f"./round-2-island-data-bottle/prices_round_2_day_{-1}.csv", sep=";", header=0)

In [ ]:
df_day_neg_1 = df_day_neg_1[['timestamp', 'ORCHIDS', 'TRANSPORT_FEES', 'EXPORT_TARIFF', 'IMPORT_TARIFF','SUNLIGHT', 'HUMIDITY']]
df_day_neg_1.loc[:, 'ORCHIDS'] = df_day_neg_1['ORCHIDS'].ewm(alpha = 0.05).mean().reset_index(drop=True)
df_day_neg_1['HUMIDITY_DIFF'] = np.abs(df_day_neg_1["HUMIDITY"] - 70)

In [ ]:
import pandas as pd
from sklearn.metrics import r2_score

def generate_returns_dataframe(df, iteration):
    columns_to_process = ['ORCHIDS', 'HUMIDITY_DIFF']
    new_df = df.copy()

    for col in columns_to_process:
        new_df = get_prev_returns(new_df, col, iteration)
        new_df = get_future_returns(new_df, col, iteration)

    return new_df

# Assuming you have your dataframe named 'df_day_neg_1'
iteration = 100

# Generate the returns DataFrame for df_day_neg_1
df_day_neg_1_returns = generate_returns_dataframe(df_day_neg_1, iteration).dropna()
df_day_neg_1_returns = df_day_neg_1_returns[df_day_neg_1_returns['HUMIDITY_DIFF'] >= 15]

# Prepare the input features (X) for prediction
X = df_day_neg_1_returns[f"HUMIDITY_DIFF_returns_from_{iteration}_its_ago"].values.reshape(-1, 1)

# Use the learned coefficient to make predictions
coefficient = 0.0471  # Replace with the coefficient from your learned equation
y_pred = coefficient * X

# Add the predicted values as a new column in the DataFrame
df_day_neg_1_returns[f"ORCHIDS_returns_in_{iteration}_its_predicted"] = y_pred

# Get the actual ORCHIDS future returns
y_true = df_day_neg_1_returns[f"ORCHIDS_returns_in_{iteration}_its"].values.reshape(-1, 1)

# Calculate the test R-squared value
test_r2 = r2_score(y_true, y_pred)

# Print the test R-squared value
print(f"Test R-squared (R^2) Value: {test_r2:.4f}")

# Print the DataFrame with the predicted values
print("\nDataFrame with Predicted Values:")
print(df_day_neg_1_returns[[f"HUMIDITY_DIFF_returns_from_{iteration}_its_ago", f"ORCHIDS_returns_in_{iteration}_its", f"ORCHIDS_returns_in_{iteration}_its_predicted"]])

In [ ]:
df_day_1_returns = generate_returns_dataframe(df_day_1, iteration).dropna()

In [ ]:
df_test = df_day_1_returns[(df_day_1_returns['timestamp'] > 127600) & (df_day_1_returns['timestamp'] < 161100)]

In [ ]:
df_test['ORCHIDS_returns_in_100_its'].max()

In [ ]:
day = -1
df = pd.read_csv(f"./round-2-island-data-bottle/prices_round_2_day_{day}.csv", sep=";", header=0)

In [ ]:
df['arb_spread'] = df['IMPORT_TARIFF'] + df['TRANSPORT_FEES']

In [ ]:
df['arb_spread'].mean()

In [ ]:
day_neg1_mean = _

In [ ]:
df['arb_spread'].max()

In [ ]:
df['arb_spread'].min()

In [ ]:
day = 0
df = pd.read_csv(f"./round-2-island-data-bottle/prices_round_2_day_{day}.csv", sep=";", header=0)

In [ ]:
df['arb_spread'] = df['IMPORT_TARIFF'] + df['TRANSPORT_FEES']

In [ ]:
df['arb_spread'].mean()

In [ ]:
day_0_mean = _

In [ ]:
df['arb_spread'].max()

In [ ]:
df['arb_spread'].min()

In [ ]:
day = 1
df = pd.read_csv(f"./round-2-island-data-bottle/prices_round_2_day_{day}.csv", sep=";", header=0)

In [ ]:
df['arb_spread'] = df['IMPORT_TARIFF'] + df['TRANSPORT_FEES']

In [ ]:
df['arb_spread'].mean()

In [ ]:
day_1_mean = _

In [ ]:
df['arb_spread'].max()

In [ ]:
df['arb_spread'].min()

In [ ]:
day_neg1_mean + day_0_mean + day_1_mean

In [ ]:
_/3